In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv("/content/drive/MyDrive/Capstone Project -  Group 8/Dataset/MetroPT3(AirCompressor).csv")
print(df.shape)
print(df.head())

(1516948, 17)
   Unnamed: 0            timestamp    TP2    TP3     H1  DV_pressure  \
0           0  2020-02-01 00:00:00 -0.012  9.358  9.340       -0.024   
1          10  2020-02-01 00:00:10 -0.014  9.348  9.332       -0.022   
2          20  2020-02-01 00:00:19 -0.012  9.338  9.322       -0.022   
3          30  2020-02-01 00:00:29 -0.012  9.328  9.312       -0.022   
4          40  2020-02-01 00:00:39 -0.012  9.318  9.302       -0.022   

   Reservoirs  Oil_temperature  Motor_current  COMP  DV_eletric  Towers  MPG  \
0       9.358           53.600         0.0400   1.0         0.0     1.0  1.0   
1       9.348           53.675         0.0400   1.0         0.0     1.0  1.0   
2       9.338           53.600         0.0425   1.0         0.0     1.0  1.0   
3       9.328           53.425         0.0400   1.0         0.0     1.0  1.0   
4       9.318           53.475         0.0400   1.0         0.0     1.0  1.0   

   LPS  Pressure_switch  Oil_level  Caudal_impulses  
0  0.0            

In [4]:
df.columns = [c.strip() for c in df.columns]
df = df.rename(columns={df.columns[0]: 'idx'})
df = df.drop(columns=['idx'])

print(df.shape)
print(df.columns.tolist())

(1516948, 16)
['timestamp', 'TP2', 'TP3', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current', 'COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses']


In [5]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(df['timestamp'].dtype)
print(df['timestamp'].min(), "to", df['timestamp'].max())

datetime64[ns]
2020-02-01 00:00:00 to 2020-09-01 03:59:50


In [6]:
# Check 1: are there any missing values anywhere?
print("Missing values per column:")
print(df.isna().sum())

# Check 2: do the 8 digital signal columns only contain 0 or 1?
digital_cols = ['COMP', 'DV_eletric', 'Towers', 'MPG', 'LPS', 'Pressure_switch', 'Oil_level', 'Caudal_impulses']
for c in digital_cols:
    print(c, "unique values:", sorted(df[c].unique()))

Missing values per column:
timestamp          0
TP2                0
TP3                0
H1                 0
DV_pressure        0
Reservoirs         0
Oil_temperature    0
Motor_current      0
COMP               0
DV_eletric         0
Towers             0
MPG                0
LPS                0
Pressure_switch    0
Oil_level          0
Caudal_impulses    0
dtype: int64
COMP unique values: [np.float64(0.0), np.float64(1.0)]
DV_eletric unique values: [np.float64(0.0), np.float64(1.0)]
Towers unique values: [np.float64(0.0), np.float64(1.0)]
MPG unique values: [np.float64(0.0), np.float64(1.0)]
LPS unique values: [np.float64(0.0), np.float64(1.0)]
Pressure_switch unique values: [np.float64(0.0), np.float64(1.0)]
Oil_level unique values: [np.float64(0.0), np.float64(1.0)]
Caudal_impulses unique values: [np.float64(0.0), np.float64(1.0)]


In [ ]:
# The 4 documented failure windows (exact start/end times from the official report)
FAILURES = [
    ("F1", pd.Timestamp("2020-04-18 00:00:00"), pd.Timestamp("2020-04-18 23:59:00")),
    ("F2", pd.Timestamp("2020-05-29 23:30:00"), pd.Timestamp("2020-05-30 06:00:00")),
    ("F3", pd.Timestamp("2020-06-05 10:00:00"), pd.Timestamp("2020-06-07 14:30:00")),
    ("F4", pd.Timestamp("2020-07-15 14:30:00"), pd.Timestamp("2020-07-15 19:00:00")),
]

# Mark which rows fall inside an actual failure, and which failure it is
df['in_failure_window'] = False
df['failure_id'] = None
for fid, start, end in FAILURES:
    mask = (df['timestamp'] >= start) & (df['timestamp'] <= end)
    df.loc[mask, 'in_failure_window'] = True
    df.loc[mask, 'failure_id'] = fid

# For every row, calculate how many minutes until the NEXT failure starts
failure_starts = [f[1] for f in FAILURES]
def time_to_next(ts):
    future = [s for s in failure_starts if s > ts]
    return (min(future) - ts).total_seconds() / 60.0 if future else np.nan

df['time_to_next_failure_min'] = df['timestamp'].apply(time_to_next)

# Prediction horizons, in minutes
HORIZONS_MIN = [5, 10, 20, 30]

# Only label samples that are:
# 1. before the next failure
# 2. not already inside a failure window
valid_pre_failure = (
    df['time_to_next_failure_min'].notna()
    & (df['time_to_next_failure_min'] > 0)
    & (~df['in_failure_window'])
)

for horizon in HORIZONS_MIN:
    label_col = f'horizon_label_{horizon}min'

    df[label_col] = (
        valid_pre_failure
        & (df['time_to_next_failure_min'] <= horizon)
    ).astype('int8')

print("Horizon label summary:")
for horizon in HORIZONS_MIN:
    label_col = f'horizon_label_{horizon}min'
    positives = int(df[label_col].sum())
    ratio = df[label_col].mean() * 100

    print(
        f"{label_col}: "
        f"{positives:,} positives "
        f"({ratio:.4f}%)"
    )

Rows inside a failure window: 29954
Positive rows (30-min warning label): 694
As a percentage: 0.0457 %


In [8]:
# --- Part 1: Oil temperature — flag when it drifts far from its own recent normal ---
window = 720  # roughly 2 hours of history, since each reading is 10 seconds apart
roll_mean = df['Oil_temperature'].rolling(window=window, min_periods=100).mean().shift(1)
roll_std = df['Oil_temperature'].rolling(window=window, min_periods=100).std().shift(1)
df['oil_temp_zscore'] = (df['Oil_temperature'] - roll_mean) / roll_std
df['oil_temp_anomaly'] = (df['oil_temp_zscore'].abs() > 3).astype(int)

# --- Part 2: Off-cycle duration — flag when the machine rests way longer than usual ---
df['is_off'] = df['Motor_current'] < 0.1
df['stretch_id'] = (df['is_off'] != df['is_off'].shift()).cumsum()

stretches = df.groupby('stretch_id').agg(
    is_off=('is_off', 'first'), start=('timestamp', 'min'), end=('timestamp', 'max')
)
stretches['duration_min'] = (stretches['end'] - stretches['start']).dt.total_seconds() / 60 + (10/60)

off_stretches = stretches[stretches['is_off']].copy()
off_stretches['baseline_mean'] = off_stretches['duration_min'].rolling(20, min_periods=5).mean().shift(1)
off_stretches['baseline_std'] = off_stretches['duration_min'].rolling(20, min_periods=5).std().shift(1)
off_stretches['duration_zscore'] = (off_stretches['duration_min'] - off_stretches['baseline_mean']) / off_stretches['baseline_std']
off_stretches['duration_anomaly'] = (off_stretches['duration_zscore'] > 3).astype(int)

anomaly_stretch_ids = set(off_stretches[off_stretches['duration_anomaly'] == 1].index)
df['off_duration_anomaly'] = df['stretch_id'].isin(anomaly_stretch_ids).astype(int)

# --- Combine both into one label: flag if EITHER signal looks abnormal ---
df['adaptive_statistical_label'] = (
    (df['oil_temp_anomaly'] == 1) | (df['off_duration_anomaly'] == 1)
).astype(int)

df = df.drop(columns=['is_off', 'stretch_id'])

print("Oil temp anomaly rows:", df['oil_temp_anomaly'].sum())
print("Off-duration anomaly rows:", df['off_duration_anomaly'].sum())
print("Combined adaptive label positive rows:", df['adaptive_statistical_label'].sum())
print("As a percentage:", round(100 * df['adaptive_statistical_label'].mean(), 4), "%")

Oil temp anomaly rows: 5805
Off-duration anomaly rows: 32995
Combined adaptive label positive rows: 38789
As a percentage: 2.557 %


In [ ]:
split_boundary = pd.Timestamp("2020-07-14 00:00:00")

df['split'] = df['timestamp'].apply(lambda t: 'train' if t < split_boundary else 'test')

print(df['split'].value_counts())
print()
print("Failures in TRAIN:", sorted(df[df['split']=='train']['failure_id'].dropna().unique()))
print("Failures in TEST:", sorted(df[df['split']=='test']['failure_id'].dropna().unique()))
print()
for horizon in HORIZONS_MIN:
    label_col = f'horizon_label_{horizon}min'

    train_positive = df.loc[
        df['split'] == 'train', label_col
    ].sum()

    test_positive = df.loc[
        df['split'] == 'test', label_col
    ].sum()

    print(
        f"{label_col}: "
        f"train={train_positive:,}, "
        f"test={test_positive:,}"
    )

split
train    1162265
test      354683
Name: count, dtype: int64

Failures in TRAIN: ['F1', 'F2', 'F3']
Failures in TEST: ['F4']

Positive warning rows in TRAIN: 513
Positive warning rows in TEST: 181


In [15]:
train_df = df[df['split'] == 'train'].drop(columns=['split'])
test_df = df[df['split'] == 'test'].drop(columns=['split'])

folder = "/content/drive/MyDrive/Capstone Project -  Group 8/Dataset/"
train_df.to_csv(folder + "metropt3_train.csv", index=False)
test_df.to_csv(folder + "metropt3_test.csv", index=False)

print("Train saved:", train_df.shape)
print("Test saved:", test_df.shape)

Train saved: (1162265, 24)
Test saved: (354683, 24)
